In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import * 

In [0]:
df= spark.read.table('fmcg2.bronze.orders')

In [0]:
display(df)

In [0]:
df.filter(col("order_qty").isNull()).count()

In [0]:
df=df.fillna(0,subset=["order_qty"])

In [0]:
df.filter(col("order_qty").isNull()).count()

In [0]:
df.select("order_placement_date").distinct().display()

In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
df = df.withColumn(
    "parsed_date",
    coalesce(
        try_to_timestamp(trim(col("order_placement_date")), lit("yyyy/MM/dd")),
        try_to_timestamp(trim(col("order_placement_date")), lit("dd-MM-yyyy")),
        try_to_timestamp(trim(col("order_placement_date")), lit("EEEE, MMMM dd, yyyy")),
        try_to_timestamp(trim(col("order_placement_date")), lit("dd/MM/yyyy")),
        try_to_timestamp(trim(col("order_placement_date")), lit("yyyy-MM-dd"))
    )
).withColumn(
    "order_placement_date",
    date_format(col("parsed_date"), "yyyy-MM-dd")
).drop("parsed_date")

In [0]:
display(df.limit(2))

In [0]:
df=df.dropDuplicates(['product_id','order_placement_date','customer_id','order_qty','product_id'])

In [0]:
df.printSchema()

In [0]:
df=df.withColumnsRenamed({'order_placement_date':'Date',
                       'customer_id':'customer_code',
                      'product_id':'product_code',
                       'order_qty': 'sold_quantity'})

In [0]:
df=df.drop("order_id")

In [0]:
display(df.limit(2))

In [0]:
df.printSchema()

In [0]:
df=df.withColumn('Date',col("Date").cast('date'))\
.withColumn('product_code',col("product_code").cast('string'))\
.withColumn('sold_quantity',col("sold_quantity").cast('bigint'))

In [0]:
df.printSchema()

In [0]:
df=df.dropDuplicates(["product_code"])

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
        .saveAsTable('fmcg2.silver.order_data_ennr')

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
        .saveAsTable('fmcg2.gold.order_data_gold')


In [0]:
from delta.tables import DeltaTable
parent= DeltaTable.forName(spark,'fmcg.gold.fact_orders')
child= spark.read.table('fmcg2.silver.order_data_ennr')

parent.alias('target').merge(child.alias('source'),'target.product_code=source.product_code')\
.whenMatchedUpdateAll()\
.whenNotMatchedInsertAll()\
    .execute()
